In [3]:
import numpy as np

# --- 1. Helical Geometry & Energy Parameters ---
M = 4          # Total number of sites
m_l = 1        # Number of helical loops
a = 1.0        # Radius of the helix (nm)
c = 1.0        # Length of the helix (nm)

# Tight-binding energetic parameters (converted to eV based on the paper's units)
t0 = 0.04             # Nearest-neighbor hopping amplitude
e0 = -6.0 * t0        # On-site energy
lambda0 = t0 / 40.0   # Spin-orbit coupling strength

# --- 2. Calculate Spatial Vectors ---
# Azimuthal angles for m = 1 to M
m_idx = np.arange(1, M + 1)
phi = 2 * np.pi * (m_idx - 1) * m_l / (M - 1)

# Position vectors r_m (Shape: M x 3)
rx = a * np.cos(phi)
ry = a * np.sin(phi)
rz = (m_idx - 1) * c / (M - 1)
r = np.vstack((rx, ry, rz)).T

# Unit bond vectors d_{m,1} pointing from site m to m+1 (Shape: M-1 x 3)
d = np.zeros((M - 1, 3))
for m in range(M - 1):
    delta_r = r[m + 1] - r[m]
    d[m] = delta_r / np.linalg.norm(delta_r)

# Chirality vectors v^{(+)}_m = d_{m,1} x d_{m+1,1} (Shape: M-2 x 3)
v = np.zeros((M - 2, 3))
for m in range(M - 2):
    v[m] = np.cross(d[m], d[m + 1])

# --- 3. Define Pauli Matrices ---
sigma_x = np.array([[0, 1], [1, 0]], dtype=complex)
sigma_y = np.array([[0, -1j], [1j, 0]], dtype=complex)
sigma_z = np.array([[1, 0], [0, -1]], dtype=complex)
I_2 = np.eye(2, dtype=complex)

# Construct V_m = v_m_x*sigma_x + v_m_y*sigma_y + v_m_z*sigma_z
V_matrices = []
for m in range(M - 2):
    Vm = v[m, 0] * sigma_x + v[m, 1] * sigma_y + v[m, 2] * sigma_z
    V_matrices.append(Vm)

# --- 4. Construct S1, S2, S3 ---
N_states = M * 2 # 2 spin states per site

# S1: Identity block matrix
S1 = np.eye(N_states, dtype=complex)

# S2: Nearest-neighbor hopping block matrix
S2 = np.zeros((N_states, N_states), dtype=complex)
for m in range(M - 1):
    row_idx = 2 * m
    col_idx = 2 * (m + 1)
    S2[row_idx:row_idx+2, col_idx:col_idx+2] = -I_2
    S2[col_idx:col_idx+2, row_idx:row_idx+2] = -I_2

# S3: Next-nearest-neighbor spin-orbit coupling block matrix
S3 = np.zeros((N_states, N_states), dtype=complex)
for m in range(M - 2):
    row_idx = 2 * m
    col_idx = 2 * (m + 2)
    Vm = V_matrices[m]
    
    # Upper diagonal is +i * V_m
    S3[row_idx:row_idx+2, col_idx:col_idx+2] = 1j * Vm
    
    # Lower diagonal is the Hermitian conjugate: -i * V_m^dagger
    S3[col_idx:col_idx+2, row_idx:row_idx+2] = -1j * Vm.conj().T

# --- 5. Assemble Final Hamiltonian ---
H_s = e0 * S1 + t0 * S2 + lambda0 * S3

In [5]:
eigenvalues, eigenvectors = np.linalg.eigh(H_s)

print(eigenvalues)

[-0.3047 -0.3047 -0.2647 -0.2647 -0.2153 -0.2153 -0.1753 -0.1753]
